# Rung 40 — G4, THE PARAM-GROUP GATE

**Does the connector's reduced learning rate actually apply?**

Path A′ trains the connector at **4e-5** — 1/5 of the LoRA LR, following
`literature/vlm-techniques/FICHAS.md:361` (Qwen2.5-VL-7B: LoRA lr 5e-5, projection-layer lr 1e-5).

A per-group LR is **plumbing, not a flag**, and this repo has already been burned by that exact
shape: **`--vit_lr` is a silent no-op without `--optimizer multimodal`**, and rung 21's `A3_vitlr`
moved two flags with nobody checking the second did anything. An arm whose declared variable never
applies produces **a null that looks like evidence** — worse than no arm.

| | | |
|---|---|---|
| **G4a** | structural | the connector sits in its own group, `lr` read back **from the optimiser**, before **and after** `.train()` |
| **G4b** | 🔑 differential | two legs, same seed, connector at 4e-5 vs 2e-4 — movement must be **~5× larger** in the second |

G4b is the one that matters: a group can exist, declare 4e-5, and be **ignored**. Only the ratio
separates "configured" from "applied".

`Qwen3.5-2B`, synthetic noise, no challenge frames — the DUA is not engaged, so UNAM is a
legitimate host. What UNAM cannot do is run the **arm**; the challenge data does not travel there.
This gate exists so that when an 80 GB card is rented, the only unknown left is the measurement.

## 1 — Environment. Same asserts as the merge gate: fail here, not after loading.

In [ ]:
import unsloth  # noqa: F401  MUST precede transformers
import importlib.metadata as md
import torch

EXPECT = {"unsloth": "2026.8.15", "unsloth_zoo": "2026.8.10"}
for pkg, want in EXPECT.items():
    got = md.version(pkg)
    assert got == want, f"{pkg} is {got}, expected {want} — a different code path than the arm runs"

assert torch.cuda.is_available(), "no CUDA"
print("gpu       ", torch.cuda.get_device_name(0))
print("free GiB  ", round(torch.cuda.mem_get_info()[0] / 2**30, 1))

## 2 — Config. Inline, per the repo spec.

In [ ]:
import logging, sys
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

TOOLS = str(Path.cwd() / "_tools")
if TOOLS not in sys.path:
    sys.path.insert(0, TOOLS)

from lr_group_gate import LRGroupGateConfig, run_lr_group_gate  # noqa: E402

cfg = LRGroupGateConfig(
    base_model="Qwen/Qwen3.5-2B",
    hf_home="/data/uaq_user/hf_cache",
    work_dir="/data/uaq_user/tmp/leo_gate40_lr",
    lora_lr=2e-4,        # the arm's LR, unchanged
    connector_lr=4e-5,   # 1/5 -- the A' variable
    control_lr=2e-4,     # differential leg: connector at the LoRA LR
    n_rows=32,
    n_steps=20,
    seed=42,
)
cfg

## 3 — Run both legs. Raises on the first failing assertion.

In [ ]:
result = run_lr_group_gate(cfg)

## 4 — Verdict.

In [ ]:
print("VERDICT:", result["verdict"])
print()
for leg in result["legs"]:
    print(f"  leg {leg['leg']:<5} connector_lr={leg['connector_lr']:<8} "
          f"sum|delta|={leg['sum_abs_delta']:.6g}  loss={leg['train_loss']:.4f}")
print()
g4b = result["G4b"]
print(f"  measured ratio : {g4b['measured_ratio']:.2f}")
print(f"  expected ratio : {g4b['expected_ratio']:.1f}")
print(f"  relative error : {g4b['relative_error']:.0%}")
if "note" in g4b:
    print("  note           :", g4b["note"])
print()
print("groups after train (low leg):", result["G4a"]["low"])